# Baseline: Logistic Regression

## Цель ноутбука
- Создать пайплайн для обработки признаков
- Обучить Logistic Regression как baseline
- Оценить качество на тестовой выборке

## Используемые данные
- Очищенный датасет с разделением train/val/test

## Основные выводы
- Модель показывает базовый уровень качества
- F1-score ниже из-за дисбаланса классов
- Требуется улучшение модели

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'Churn_Modelling.csv'

drop_columns = ['RowNumber', 'CustomerId', 'Surname']
target = 'Exited'

numeric_features = [
    'CreditScore', 'Age', 'Tenure', 'Balance',
    'NumOfProducts', 'EstimatedSalary'
]

categorical_features = ['Geography', 'Gender']
binary_features = ['HasCrCard', 'IsActiveMember']

all_features = numeric_features + categorical_features + binary_features

df = pd.read_csv(DATA_PATH)
df_clean = df.drop(columns=drop_columns).drop_duplicates().copy()

X = df_clean[all_features]
y = df_clean[target]

## 1. Разделение данных

In [ ]:
random_state = 42
test_size = 0.15
val_size = 0.15

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=test_size,
    stratify=y,
    random_state=random_state
)

val_ratio_from_train = val_size / (1 - test_size)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=val_ratio_from_train,
    stratify=y_train_full,
    random_state=random_state
)

print('Train shape:', X_train.shape)
print('Val shape:', X_val.shape)
print('Test shape:', X_test.shape)
print('\nДоля Exited в train:', y_train.mean())
print('Доля Exited в val:', y_val.mean())
print('Доля Exited в test:', y_test.mean())

## 2. Создание пайплайна

In [ ]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

bin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features),
    ('bin', bin_pipe, binary_features)
])

baseline_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

print('Пайплайн создан')
baseline_model

## 3. Обучение модели

In [ ]:
baseline_model.fit(X_train, y_train)
print('Модель обучена')

## 4. Оценка качества

In [ ]:
baseline_pred = baseline_model.predict(X_test)
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)

print('=' * 40)
print('Baseline Logistic Regression')
print('=' * 40)
print(f'Accuracy:  {baseline_accuracy:.4f}')
print(f'F1-score:  {baseline_f1:.4f}')
print(f'ROC-AUC:   {baseline_auc:.4f}')
print('=' * 40)

In [ ]:
print('\nClassification Report:')
print(classification_report(y_test, baseline_pred, target_names=['Не ушли', 'Ушли']))

In [ ]:
cm = confusion_matrix(y_test, baseline_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix: Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 5. Выводы по Baseline

- Logistic Regression показывает базовый уровень качества
- Проблема: низкий F1-score из-за дисбаланса классов
- Модель плохо распознаёт класс "Ушли" (низкий recall)
- Требуется использовать более сложные модели (CatBoost, XGBoost)